In [1]:
from sentence_transformers import SentenceTransformer, util
from FlagEmbedding import FlagReranker
import torch

# ===== DEVICE =====
device = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16 = device == "cuda"

print(f"Using device: {device}")

# ===== MODELE =====
embed_model = SentenceTransformer("BAAI/bge-m3", device=device)

if use_fp16:
    embed_model.half()

embed_model.max_seq_length = 8192

reranker = FlagReranker(
    "BAAI/bge-reranker-v2-m3",
    use_fp16=use_fp16
)

if device == "cuda":
    reranker.model.to(device)


# ===== FUNKCJA ZAMIAST API =====
def compute_similarity(query, documents):
    if not query or not isinstance(documents, list):
        raise ValueError("query musi być stringiem, documents listą")

    if len(documents) == 0:
        return []

    # ===== BEFORE =====
    query_emb = embed_model.encode(
        query,
        convert_to_tensor=True,
        normalize_embeddings=True,
        device=device
    )

    doc_embs = embed_model.encode(
        documents,
        convert_to_tensor=True,
        normalize_embeddings=True,
        device=device,
        batch_size=32
    )

    scores_before = util.cos_sim(query_emb, doc_embs)[0]

    # ===== AFTER =====
    pairs = [[query, doc] for doc in documents]
    scores_after = reranker.compute_score(pairs)

    # ===== OUTPUT =====
    results = []
    for doc, s_before, s_after in zip(documents, scores_before, scores_after):
        results.append({
            "document": doc,
            "score_before": float(s_before),
            "score_after": float(s_after)
        })

    return results


# ===== PRZYKŁAD UŻYCIA =====
query = "Co to jest sztuczna inteligencja?"
documents = [
    "AI to dziedzina informatyki zajmująca się uczeniem maszynowym.",
    "Pies to zwierzę domowe.",
    "Modele językowe analizują tekst."
]

results = compute_similarity(query, documents)

for r in results:
    print(r)

c:\Users\admin\Downloads\chunker-analyzer\chunker-analyzer-v8\similarity_backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 999.36it/s]

{'document': 'AI to dziedzina informatyki zajmująca się uczeniem maszynowym.', 'score_before': 0.576171875, 'score_after': 0.6669921875}
{'document': 'Pies to zwierzę domowe.', 'score_before': 0.37158203125, 'score_after': -11.03125}
{'document': 'Modele językowe analizują tekst.', 'score_before': 0.3759765625, 'score_after': -8.59375}



c:\Users\admin\Downloads\chunker-analyzer\chunker-analyzer-v8\similarity_backend\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:2888: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
